## Fact Bronze Layer - Streaming Ingestion

Ingesting transactional fact data from raw landing zone using Auto Loader:
* **brz_order_items** - Order line items (transactional grain)
* **brz_order_returns** - Product returns
* **brz_order_shipments** - Shipping details

**Key Features**:
- Auto Loader with explicit schemas
- Unity Catalog volumes for checkpoints
- Metadata tracking (_ingested_at, _source_file)

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, DateType, TimestampType, StringType, IntegerType, LongType

# Defining paths
dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
catalog_name = dbutils.widgets.get("catalog_name")
checkpoint_base = f"/Volumes/{catalog_name}/raw/checkpoints"
landing_base = f"/Volumes/{catalog_name}/raw/raw_landing"

print(f"Catalog: {catalog_name}")
print(f"Checkpoint base: {checkpoint_base}")
print(f"Landing base: {landing_base}")

# Defining schemas for each fact table
schemas = {
    "order_items": StructType([
        StructField("dt", DateType(), True),
        StructField("order_ts", TimestampType(), True),
        StructField("customer_id", StringType(), True),
        StructField("order_id", IntegerType(), False),
        StructField("item_seq", IntegerType(), False),
        StructField("product_id", LongType(), True),
        StructField("quantity", StringType(), True),
        StructField("unit_price_currency", StringType(), True),
        StructField("unit_price", StringType(), True),
        StructField("discount_pct", StringType(), True),
        StructField("tax_amount", IntegerType(), True),
        StructField("channel", StringType(), True),
        StructField("coupon_code", StringType(), True)
    ]),
    "order_returns": StructType([
        StructField("return_ts", TimestampType(), True),
        StructField("order_dt", DateType(), True),
        StructField("order_id", IntegerType(), False),
        StructField("reason", StringType(), True)
    ]),
    "order_shipments": StructType([
        StructField("order_dt", DateType(), True),
        StructField("shipment_id", StringType(), False),
        StructField("order_id", IntegerType(), False),
        StructField("carrier", StringType(), True)
    ])
}

print(f"\nSchemas defined for {len(schemas)} fact tables")

In [0]:
# Ingest fact tables using Auto Loader with streaming
for table_name, schema in schemas.items():
    print(f"\n{'='*60}")
    print(f"Processing fact table: {table_name}")
    print(f"{'='*60}")

    checkpoint_data = f"{checkpoint_base}/brz_{table_name}/data"
    checkpoint_schema = f"{checkpoint_base}/brz_{table_name}/schema"
    source_path = f"{landing_base}/{table_name}/"

    print(f"Source: {source_path}")
    print(f"Checkpoint: {checkpoint_data}")

    # Read data from landing zone using Auto Loader
    df_stream = (spark.readStream
                 .format("cloudFiles")
                 .option("cloudFiles.format", "csv")
                 .option("cloudFiles.schemaLocation", checkpoint_schema)
                 .option("cloudFiles.schemaEvolutionMode", "rescue")
                 .option("header", "true")
                 .option("rescuedDataColumn", "_rescued_data")
                 .option("cloudFiles.includeExistingFiles", "true")
                 .option("pathGlobFilter", "*.csv")
                 .schema(schema)
                 .load(source_path)
    )

    # Adding metadata columns
    df_final = df_stream.select(
        "*",
        F.current_timestamp().alias("_ingested_at"),
        F.col("_metadata.file_path").alias("_source_file")
    )

    # Write stream to bronze Delta table
    (df_final.writeStream
            .format("delta")
            .option("checkpointLocation", checkpoint_data)
            .option("mergeSchema", "true")
            .outputMode("append")
            .trigger(availableNow=True)
            .toTable(f"{catalog_name}.bronze.brz_{table_name}")
            .awaitTermination()
    )
    print(f"✓ Completed: {table_name}")

print(f"All fact tables processed successfully!")

In [0]:
# Check row counts for all bronze fact tables
tables = ['brz_order_items', 'brz_order_returns', 'brz_order_shipments']

for table in tables:
    try:
        count = spark.table(f"{catalog_name}.bronze.{table}").count()
        print(f"{table:25} : {count:,} rows")
    except Exception as e:
        print(f"{table:25} : Table not found or error")